In [2]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

C:\Users\This PC\AppData\Roaming\Python\Python313\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
train_data = pd.read_csv(r"C:\Users\This PC\Downloads\samsum-train.csv")
val_data = pd.read_csv(r"C:\Users\This PC\Downloads\samsum-validation.csv")

In [4]:
train_data.head()
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [5]:
# random sampling
train_data = train_data.sample(n=1000,random_state=42).reset_index(drop = True)
val_data = val_data.sample(n=300,random_state=42).reset_index(drop=True)

## Data Pre-Processing

In [6]:
import re

def clean_data(text):
    text = re.sub(r"\n\r"," ",text) # lines
    text = re.sub(r"\s+"," ",text) # spaces
    text = re.sub(r"<.*?>"," ",text) # html tags <p> <h1>
    text = text.strip().lower() #  removes extra spaces from the beginning and end,converts the entire string to lowercase
    return text

In [7]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

## Tokenize

In [8]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
# raw data => tokenized inputs for fine-tuning 
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=312,truncation=True)
    target = tokenizer(data["summary"],padding="max_length", max_length=100,truncation=True)

    inputs["labels"] = target["input_ids"]
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize,axis = 1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [12]:
train_dataset[1]

{'input_ids': [6234, 10, 78, 405, 1321, 214, 116, 8, 6093, 19, 352, 12, 1837, 58, 16585, 10, 12050, 6, 150, 6, 68, 133, 310, 114, 12, 5, 3, 1050, 2494, 10, 3, 23, 278, 31, 17, 317, 3, 23, 31, 26, 36, 1638, 16, 48, 5, 6234, 10, 3, 63, 58, 3, 1050, 2494, 10, 2492, 66, 8, 1717, 11, 3224, 3640, 7, 656, 140, 1227, 19974, 5, 16585, 10, 78, 25, 31, 60, 78, 7569, 58, 6234, 10, 3, 75, 31, 2157, 55, 3, 7, 52, 7, 120, 58, 3, 1050, 2494, 10, 3, 63, 413, 5, 141, 8, 337, 589, 437, 3, 23, 47, 3, 9, 861, 5, 16585, 10, 2087, 34, 31, 7, 97, 12, 483, 34, 58, 6234, 10, 17945, 55, 428, 34, 3, 9, 653, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [13]:
# input ids - dialogue => token ids

# 1 => EOS

# attention mask
# labels = target => summary token

In [14]:
len(train_dataset[0]["input_ids"])
type(train_dataset)
type(val_dataset)

list

## Working with our model

In [15]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [16]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device:",device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    
    num_train_epochs=6,
    weight_decay=0.01,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    
    warmup_steps=500 
    # 0 => lr default
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset)

In [19]:
# train the model
trainer.train()

C:\Users\This PC\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,5.748295
2,No log,0.741226
3,No log,0.578317
4,3.827276,0.554038
5,3.827276,0.542179
6,3.827276,0.540575


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\This PC\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

C:\Users\This PC\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\This PC\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=2.742030314127604, metrics={'train_runtime': 42955.7289, 'train_samples_per_second': 0.14, 'train_steps_per_second': 0.017, 'total_flos': 494843461632000.0, 'train_loss': 2.742030314127604, 'epoch': 6.0})

In [22]:
# model load => fine-tune => save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [23]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core lodgic for summarization

In [31]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)  # clean

    # tokenize 
    inputs = tokenizer(dialogue,padding = "max_length",max_length=312,truncation=True,return_tensors ="pt").to(device)

    # generate the summary => token ids
    targets = model.generate(input_ids = inputs["input_ids"],attention_mask=inputs["attention_mask"],max_length=110, num_beams=4,early_stopping=True)

    # decode our output
    summary = tokenizer.decode(targets[0],skip_special_tokens = True)
    return summary

In [32]:
test_dialogue = """ Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary:",summary)

Summary: reporter is investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
